# LLM / GPU Cost Optimizer — Core Simulator

**Goal:** experimentally prove whether smart model routing + prompt caching can reduce cost **without silently degrading quality**.

This notebook intentionally starts with mocked model tiers so the routing policy can be tested quickly and reproducibly before wiring real LLM APIs.

## Core hypothesis

> Route each request to the **cheapest model that is predicted to satisfy the required quality threshold**, and escalate uncertain / failed cases.

We will compare:
1. **Always Frontier** — every request uses the expensive model.
2. **Naive Difficulty Router** — easy → cheap, hard → frontier.
3. **Quality-Aware Router** — choose the cheapest model whose predicted quality clears the threshold.
4. **Quality-Aware + Cache** — same router, plus repeated-context prompt caching.

The final dashboard should answer the judge's key question:
**How much money did we save, and what happened to quality?**

In [ ]:
import hashlib
import math
import random
from dataclasses import dataclass
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

random.seed(42)
np.random.seed(42)

print("Environment ready")

## 1. Benchmark design

Do **not** benchmark only easy prompts. That makes a router look artificially good.

Use a mixed task set with categories that stress different capabilities: factual QA, extraction, coding, reasoning, multi-step reasoning, long-context synthesis, and ambiguous prompts.

In [ ]:
benchmark = [
    # id, category, difficulty, prompt_tokens, context_id, gold_quality
    ("q01", "simple_qa", 1, 180, "policy_A", 1.0),
    ("q02", "simple_qa", 1, 160, "policy_A", 1.0),
    ("q03", "extraction", 2, 260, "invoice_schema", 1.0),
    ("q04", "extraction", 2, 240, "invoice_schema", 1.0),
    ("q05", "coding", 3, 420, "python_style", 1.0),
    ("q06", "coding", 4, 600, "python_style", 1.0),
    ("q07", "reasoning", 4, 520, "math_rules", 1.0),
    ("q08", "reasoning", 5, 650, "math_rules", 1.0),
    ("q09", "multistep", 5, 900, "research_context", 1.0),
    ("q10", "long_context", 4, 1800, "research_context", 1.0),
    ("q11", "long_context", 5, 2200, "research_context", 1.0),
    ("q12", "ambiguous", 3, 350, "support_policy", 1.0),
    ("q13", "simple_qa", 1, 150, "policy_A", 1.0),
    ("q14", "coding", 3, 450, "python_style", 1.0),
    ("q15", "multistep", 5, 1000, "research_context", 1.0),
]

df = pd.DataFrame(
    benchmark,
    columns=["id", "category", "difficulty", "prompt_tokens", "context_id", "gold_quality"]
)

df

## 2. Mock model tiers

We model two tiers:
- **Small**: cheap, fast, but quality falls faster as task difficulty rises.
- **Frontier**: expensive, strong, and more robust on difficult tasks.

The important part is that quality is simulated explicitly. A cost-only router can therefore be punished when it routes hard requests badly.

In [ ]:
@dataclass(frozen=True)
class ModelTier:
    name: str
    input_cost_per_1k: float
    output_cost_per_1k: float
    base_quality: float
    difficulty_penalty: float
    speed_ms: int

SMALL = ModelTier(
    name="small",
    input_cost_per_1k=0.00030,
    output_cost_per_1k=0.00120,
    base_quality=0.985,
    difficulty_penalty=0.120,
    speed_ms=350,
)

FRONTIER = ModelTier(
    name="frontier",
    input_cost_per_1k=0.00600,
    output_cost_per_1k=0.01800,
    base_quality=0.998,
    difficulty_penalty=0.028,
    speed_ms=950,
)

MODELS = {m.name: m for m in [SMALL, FRONTIER]}

def estimate_output_tokens(row):
    # Simple simulator; real system would observe actual completion length.
    return max(80, int(row.prompt_tokens * 0.35))

def model_quality(model, difficulty):
    # Keep quality in [0, 1].
    q = model.base_quality - model.difficulty_penalty * max(0, difficulty - 1)
    return float(np.clip(q, 0.0, 1.0))

def call_mock_model(row, model_name):
    model = MODELS[model_name]
    quality = model_quality(model, row.difficulty)

    # Small stochastic variation to resemble real responses.
    noise = np.random.normal(0, 0.006)
    observed_quality = float(np.clip(quality + noise, 0.0, 1.0))

    output_tokens = estimate_output_tokens(row)
    input_tokens = row.prompt_tokens

    cost = (
        input_tokens / 1000 * model.input_cost_per_1k
        + output_tokens / 1000 * model.output_cost_per_1k
    )

    return {
        "model": model.name,
        "quality": observed_quality,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "cost": cost,
        "latency_ms": model.speed_ms + int(np.random.normal(0, 30)),
    }

for name in MODELS:
    print(name, model_quality(MODELS[name], 5))

## 3. Prompt-cache simulator

We simulate caching the repeated **system/context prefix** rather than blindly caching the entire answer.

For a cache hit, only the uncached request tokens are charged. The exact discount is configurable because real providers expose different caching semantics.

In [ ]:
class PromptCache:
    def __init__(self, discount=0.10):
        self.discount = discount
        self.store = {}
        self.hits = 0
        self.misses = 0

    def key(self, context_id):
        return hashlib.sha256(context_id.encode()).hexdigest()

    def lookup(self, context_id):
        k = self.key(context_id)
        if k in self.store:
            self.hits += 1
            return True
        self.misses += 1
        self.store[k] = True
        return False

def apply_cache_to_cost(row, model_name, cache_enabled, cache):
    model = MODELS[model_name]
    output_tokens = estimate_output_tokens(row)

    hit = cache_enabled and cache.lookup(row.context_id)

    effective_input_tokens = row.prompt_tokens
    input_rate_multiplier = 1.0

    # Simulate cached prefix discount only after the first occurrence.
    if hit:
        input_rate_multiplier = cache.discount

    cost = (
        effective_input_tokens / 1000 * model.input_cost_per_1k * input_rate_multiplier
        + output_tokens / 1000 * model.output_cost_per_1k
    )

    return cost, hit

## 4. Three routing policies

**Naive router:** use difficulty alone.

**Quality-aware router:** estimate each model's expected quality and choose the cheapest model whose prediction clears `QUALITY_THRESHOLD`.

**Safety valve:** if the small model actually underperforms during an optional evaluator pass, escalate to frontier. This is the most important feature for the judging criterion.

In [ ]:
QUALITY_THRESHOLD = 0.94

def difficulty_router(row):
    return "small" if row.difficulty <= 3 else "frontier"

def quality_aware_router(row, threshold=QUALITY_THRESHOLD):
    candidates = sorted(MODELS.values(), key=lambda m: m.input_cost_per_1k + m.output_cost_per_1k)

    for model in candidates:
        expected = model_quality(model, row.difficulty)
        if expected >= threshold:
            return model.name

    return "frontier"

def quality_gap(row):
    return model_quality(FRONTIER, row.difficulty) - model_quality(SMALL, row.difficulty)

print("Router examples:")
for _, row in df.iterrows():
    print(row.id, row.difficulty, difficulty_router(row), quality_aware_router(row), round(quality_gap(row), 3))

## 5. Run the full simulation

We keep the **same benchmark** for every policy so the comparison is apples-to-apples.

In [ ]:
def run_policy(policy_name, cache_enabled=False):
    cache = PromptCache(discount=0.10)
    rows = []

    for _, row in df.iterrows():
        if policy_name == "always_frontier":
            selected = "frontier"
        elif policy_name == "difficulty_router":
            selected = difficulty_router(row)
        elif policy_name in {"quality_router", "quality_router_cache"}:
            selected = quality_aware_router(row)
        else:
            raise ValueError(policy_name)

        result = call_mock_model(row, selected)

        # In cache mode, recompute only cost; quality is unchanged.
        cached_cost, cache_hit = apply_cache_to_cost(
            row, selected, cache_enabled, cache
        )

        if cache_enabled:
            result["cost"] = cached_cost

        result.update({
            "request_id": row.id,
            "category": row.category,
            "difficulty": row.difficulty,
            "context_id": row.context_id,
            "cache_hit": cache_hit,
            "policy": policy_name,
        })
        rows.append(result)

    return pd.DataFrame(rows)

experiments = pd.concat([
    run_policy("always_frontier", cache_enabled=False),
    run_policy("difficulty_router", cache_enabled=False),
    run_policy("quality_router", cache_enabled=False),
    run_policy("quality_router_cache", cache_enabled=True),
], ignore_index=True)

experiments.head()

## 6. Executive scoreboard

In [ ]:
baseline = experiments[experiments.policy == "always_frontier"]
baseline_cost = baseline.cost.sum()
baseline_quality = baseline.quality.mean()

summary = (
    experiments.groupby("policy")
    .agg(
        total_cost=("cost", "sum"),
        avg_quality=("quality", "mean"),
        p10_quality=("quality", lambda s: s.quantile(0.10)),
        avg_latency_ms=("latency_ms", "mean"),
    )
    .reset_index()
)

summary["cost_saved_vs_baseline"] = baseline_cost - summary["total_cost"]
summary["cost_reduction_pct"] = 100 * summary["cost_saved_vs_baseline"] / baseline_cost
summary["quality_delta_vs_baseline"] = summary["avg_quality"] - baseline_quality

summary

## 7. Quality guardrail: don't celebrate a cheap bad answer

This view checks every policy for the worst-performing requests. In a real system, the evaluator could be an LLM judge, exact-match / structured checks, unit tests, or domain-specific validators.

In [ ]:
quality_view = experiments.sort_values(["policy", "quality"])

quality_view[
    ["policy", "request_id", "category", "difficulty", "model", "quality", "cost"]
].head(20)

## 8. Cost vs quality frontier

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.scatter(summary["total_cost"], summary["avg_quality"], s=100)

for _, r in summary.iterrows():
    ax.annotate(r["policy"], (r["total_cost"], r["avg_quality"]), xytext=(6, 6),
                textcoords="offset points")

ax.set_xlabel("Total benchmark cost")
ax.set_ylabel("Average quality")
ax.set_title("Quality–Cost Frontier")
ax.grid(alpha=0.25)
plt.show()


    ## 9. High-value experiment: vary the quality threshold

    This is where the project becomes more interesting than a fixed heuristic.

    A threshold of `0.90` may maximize savings.
    A threshold of `0.97` may route more work to the frontier.
    The useful result is the **Pareto frontier** between cost and quality.
    

In [ ]:
threshold_rows = []

for threshold in np.arange(0.90, 0.999, 0.005):
    total_cost = 0.0
    qualities = []

    for _, row in df.iterrows():
        model_name = quality_aware_router(row, threshold)
        r = call_mock_model(row, model_name)
        total_cost += r["cost"]
        qualities.append(r["quality"])

    threshold_rows.append({
        "threshold": threshold,
        "cost": total_cost,
        "quality": np.mean(qualities),
    })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.head()

In [ ]:
fig, ax1 = plt.subplots(figsize=(9, 5))

ax1.plot(threshold_df["threshold"], threshold_df["cost"], marker="o")
ax1.set_xlabel("Minimum predicted quality threshold")
ax1.set_ylabel("Total cost")

ax2 = ax1.twinx()
ax2.plot(threshold_df["threshold"], threshold_df["quality"], marker="s")
ax2.set_ylabel("Average quality")

ax1.set_title("Routing Policy Sensitivity")
plt.show()


    ## 10. What we should build next

    ### Core MVP
    - FastAPI middleware with one `/generate` endpoint.
    - Request feature extractor: token count, task type, context length, code markers, structured-output requirement, ambiguity signals.
    - Two model adapters behind a common interface.
    - Exact prefix cache first; semantic cache later.
    - Quality evaluator with task-specific checks.
    - Automatic escalation from small → frontier on low-confidence or failed validation.
    - Logging for model chosen, cache hit, tokens, cost, quality, latency, and escalation reason.

    ### “Winning demo” features
    - A **before/after cost dashboard**.
    - A **quality guardrail dashboard** showing no meaningful quality regression.
    - A live **routing explanation**: “Small model selected because predicted quality 0.97 > threshold 0.94; expected savings 82%.”
    - A **Pareto curve** showing the cost/quality trade-off.
    - A **what-if budget slider** that changes the routing threshold.
    - An **emergency fallback**: frontier model when confidence is low.
    - Cache analytics: hit rate, cached tokens, estimated savings.

    ### Research question
    Instead of claiming “80% savings,” measure:
    `cost_reduction = 1 - optimized_cost / frontier_only_cost`

    while simultaneously reporting:
    `quality_delta = optimized_quality - frontier_quality`

    The strongest demo result is therefore:
    **large cost reduction + near-zero quality delta + transparent escalation behavior.**
    


    ## 11. Suggested final architecture

    ```text
    Client
       |
       v
    FastAPI Middleware
       |
       +--> Request Fingerprint / Cache Lookup
       |
       +--> Feature Extractor
       |       - difficulty
       |       - context length
       |       - task class
       |       - structured-output need
       |       - confidence
       |
       +--> Quality-Aware Router
       |       |
       |       +--> Small Model
       |       |
       |       +--> Frontier Model
       |
       +--> Validator / Quality Evaluator
       |       |
       |       +--> PASS --> return
       |       |
       |       +--> FAIL / UNCERTAIN --> escalate to frontier
       |
       +--> Metrics Store
               - cost
               - tokens
               - latency
               - cache hit rate
               - quality
               - escalations
    ```
    